In [ ]:
import sys

from pathlib import Path
from typing import Tuple

module_path = str(Path("..").resolve())
if module_path not in sys.path:
    sys.path.append(module_path)

import utils.data_io as data_io
import utils.vis as vis

from config_handlers.rk_handler import RKHandler

In [ ]:
rk_hdl = RKHandler()

model_list = rk_hdl.get_model_list()
rk_hdl.set_curr_model(model_list[0])
ds_list = rk_hdl.get_ds_list()

all_ds = ds_list * 4
all_models = model_list * 4
all_models.sort()

[Errno 17] File exists: 'behaviour_charts'


In [ ]:
def load_data(source: str = "greedy") -> Tuple[dict, dict]:
    samples = {m: {ds: {} for ds in ds_list} for m in model_list}
    summaries = {m: {ds: {} for ds in ds_list} for m in model_list}

    for model, ds in zip(all_models, all_ds):
        if source == "nucleus":
            base_fp = Path("raw_stats", "similarity_nucleus")
        else:
            base_fp = Path("raw_stats", "similarity")

        samples[model][ds] = data_io.load_json(
            base_fp.joinpath(model, ds, "samples.json")
        )
        summaries[model][ds] = data_io.load_json(
            base_fp.joinpath(model, ds, "summary.json")
        )

    return samples, summaries

Check all similarity values

In [4]:
# det: data obtained with greedy decoding
# sam: data obtained with nucleus sampling decoding
sources = ["greedy", "nucleus"]
measures = ["ged", "iou_nodes", "iou_edges", "cosine"]
samples_greedy, summaries_greedy = load_data(source="greedy")
samples_nucleus, summaries_nuclues = load_data(source="nucleus")

for model, ds in zip(all_models, all_ds):
    print("=" * 10, f"{model.upper()} + {ds.upper()}", "=" * 10)
    print("\t\t = Greedy = \t = Nucleus =")

    # Get GED
    avg_ged_det = summaries_greedy[model][ds]["ged_avg"]
    avg_ged_sam = summaries_nuclues[model][ds]["ged_avg"]
    print(f"Avg. GED: \t {avg_ged_det:.2f} \t\t {avg_ged_sam:.2f}")
    print("-" * 50)

    # Get IOU
    avg_iou_nodes_det = summaries_greedy[model][ds]["iou"]["iou_nodes_avg"]
    avg_iou_edges_det = summaries_greedy[model][ds]["iou"]["iou_edges_avg"]
    avg_iou_nodes_sam = summaries_nuclues[model][ds]["iou"]["iou_nodes_avg"]
    avg_iou_edges_sam = summaries_nuclues[model][ds]["iou"]["iou_edges_avg"]
    print(f"Avg. IOU (n): \t {avg_iou_nodes_det:.2f} \t\t {avg_iou_edges_det:.2f}")
    print(f"Avg. IOU (e): \t {avg_iou_edges_sam:.2f} \t\t {avg_iou_edges_sam:.2f}")
    print("-" * 50)

    # Get Cosine
    avg_cosine_det = summaries_greedy[model][ds]["cosine_avg"]
    avg_cosine_sam = summaries_nuclues[model][ds]["cosine_avg"]
    print(f"Avg. Cosine: \t {avg_cosine_det:.2f} \t\t {avg_cosine_sam:.2f}")
    print("-" * 50)

========== INTERNVL2 + LLAVA-BENCH ==========
		 = Greedy = 	 = Nucleus =
Avg. GED: 	 27.10 		 27.26
--------------------------------------------------
Avg. IOU (n): 	 0.05 		 0.02
Avg. IOU (e): 	 0.02 		 0.02
--------------------------------------------------
Avg. Cosine: 	 0.56 		 0.57
--------------------------------------------------
========== INTERNVL2 + MMBENCH ==========
		 = Greedy = 	 = Nucleus =
Avg. GED: 	 22.28 		 22.63
--------------------------------------------------
Avg. IOU (n): 	 0.09 		 0.02
Avg. IOU (e): 	 0.02 		 0.02
--------------------------------------------------
Avg. Cosine: 	 0.60 		 0.59
--------------------------------------------------
========== INTERNVL2 + SEED ==========
		 = Greedy = 	 = Nucleus =
Avg. GED: 	 22.39 		 22.50
--------------------------------------------------
Avg. IOU (n): 	 0.08 		 0.04
Avg. IOU (e): 	 0.03 		 0.03
--------------------------------------------------
Avg. Cosine: 	 0.54 		 0.55
------------------------------------------

In [5]:
# Visualising measures
sim_charts_dir = Path("similarity_charts")
data_io.make_dir(sim_charts_dir)
for s in sources:
    for measure in measures:
        out_path = sim_charts_dir.joinpath(f"{measure}_{s}.pdf")
        if s == "greedy":
            vis.plot_similarity(samples_greedy, out_path, measure_key=measure)
        elif s == "nucleus":
            vis.plot_similarity(samples_nucleus, out_path, measure_key=measure)
        else:
            print("Problem!")

[Errno 17] File exists: 'similarity_charts'


Proceeding with data from greedy decoding.